# 06 — LanceDB: industry-wise schema + Word/PDF upload with LangChain

Everything is in this notebook: schema, embedding, LangChain loading, ingestion gates, LanceDB store, tests.

| # | Section |
|---|---|
| 0 | Setup (uv `.venv` kernel) |
| 1 | Schema: base record + 8 industry models (Pydantic = single source of truth) |
| 2 | Embedding model (pinned, cached by content hash) |
| 3 | Load Word/PDF with **LangChain** + attach planned metadata from the manifest |
| 4 | Ingestion gates: source → purpose → classification → ACL → retention → sensitive scan → section chunking → validate |
| 5 | LanceDB store: one table per industry, PyArrow schema generated from the Pydantic model, scalar indexes, `CURRENT` pointer |
| 6 | Upload all industry documents (bad docs → quarantine) |
| 7 | Idempotency · 8 Staging → promote · 9 Safe search · 10 Isolation · 11 Update · 12 Delete · 13 Metrics |

**LanceDB is embedded** (`lancedb.connect("data/lancedb")`, no server, no auth). System of record = a per-notebook SQLite file `data/sor/lancedb.db` (stdlib `sqlite3`), *not* the Postgres `sources` table of notebook 01.

Inputs: `data/docs/<industry>/*.pdf|*.docx` + `manifest.jsonl` (made by `scripts/make_docs.py`, all content fake).
Design: [docs/PLAN.md](../docs/PLAN.md) · Setup: [SETUP.md](../SETUP.md)

## 0. Setup

In [ ]:
import os, re, json, math, time, hashlib, statistics, types, warnings
from datetime import date, datetime, timezone
from pathlib import Path
from typing import ClassVar, Literal, Union, get_args, get_origin
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # model already cached locally; delete this line on a new machine
DOCS, SAMPLES, RESULTS = ROOT / "data" / "docs", ROOT / "data" / "samples", ROOT / "data" / "results"
pd.set_option("display.max_colwidth", 90)

import sys; print("python:", sys.executable)

## 1. Schema — one record = one retrievable chunk

Base groups from the blueprint: **identity, provenance, content/vector, access/context, lifecycle**. Each industry adds its own typed fields and rules. Validation is the last gate: anything invalid never reaches the index.

In [ ]:
SCHEMA_VERSION = 1
TOKEN = r"^tok_[a-z0-9]+$"          # tokenized identifier, never a raw ID
Status = Literal["staged", "active", "superseded", "deleted"]
Classification = Literal["public", "internal", "confidential", "restricted"]


def make_chunk_id(tenant_id, source_id, source_version, chunk_no) -> str:
    """Deterministic -> idempotent upserts."""
    return hashlib.sha256(f"{tenant_id}|{source_id}|{source_version}|{chunk_no}".encode()).hexdigest()


def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


class BaseChunk(BaseModel):
    model_config = ConfigDict(extra="forbid")
    INDUSTRY: ClassVar[str] = ""
    # identity
    chunk_id: str
    tenant_id: str = Field(min_length=1)
    industry: str
    entity_type: str
    # provenance
    source_id: str
    source_version: str
    source_file: str
    chunk_no: int = Field(ge=0)
    section: str | None = None
    content_hash: str
    parent_id: str | None = None
    # content / vector
    text: str = Field(min_length=1)
    embedding: list[float]
    embedding_model: str
    language: str = "en"
    # access / context
    classification: Classification
    allowed_groups: list[str] = Field(min_length=1)        # fail closed: no ACL -> reject
    jurisdiction: str
    valid_from: date
    valid_to: date | None = None
    # lifecycle
    retention_until: date
    status: Status = "staged"
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

    @model_validator(mode="after")
    def _base_guardrails(self):
        if self.chunk_id != make_chunk_id(self.tenant_id, self.source_id, self.source_version, self.chunk_no):
            raise ValueError("chunk_id is not the stable hash of tenant|source|version|chunk_no")
        if self.content_hash != content_hash(self.text):
            raise ValueError("content_hash does not match text")
        if not self.embedding or not all(math.isfinite(x) for x in self.embedding):
            raise ValueError("embedding empty or non-finite")
        if self.valid_to and self.valid_to < self.valid_from:
            raise ValueError("valid_to before valid_from")
        if self.retention_until < date.today():
            raise ValueError("retention_until already passed")
        return self


BASE_FIELDS = frozenset(BaseChunk.model_fields)

In [ ]:
class HealthcareChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "healthcare"
    industry: Literal["healthcare"] = "healthcare"
    patient_ref_token: str = Field(pattern=TOKEN)
    record_type: str
    provider: str
    encounter_date: date
    consent_scope: Literal["treatment", "care_coordination", "research"]


class BankingChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "banking"
    industry: Literal["banking"] = "banking"
    customer_ref_token: str | None = Field(default=None, pattern=TOKEN)
    product: str
    account_type: str
    document_type: str
    effective_date: date


class InsuranceChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "insurance"
    industry: Literal["insurance"] = "insurance"
    policy_ref: str = Field(pattern=TOKEN)
    claim_ref: str | None = Field(default=None, pattern=TOKEN)
    line_of_business: str
    event_date: date | None = None
    claim_status: Literal["open", "closed", "n/a"]


class LegalChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "legal"
    industry: Literal["legal"] = "legal"
    matter_id: str
    client_ref: str = Field(pattern=TOKEN)
    privilege_status: Literal["none", "privileged", "work_product"]
    document_type: str
    effective_date: date

    @model_validator(mode="after")
    def _privilege(self):
        if self.privilege_status != "none" and self.classification != "restricted":
            raise ValueError("privileged/work-product content must be classified 'restricted'")
        return self


class RetailChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "retail"
    industry: Literal["retail"] = "retail"
    product_id: str
    sku: str
    brand: str
    locale: str
    channel: Literal["online", "store", "omni"]
    price_valid_from: date | None = None
    price_valid_to: date | None = None
    inventory_region: str


class ManufacturingChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "manufacturing"
    industry: Literal["manufacturing"] = "manufacturing"
    asset_id: str
    site_id: str
    equipment_type: str
    safety_class: Literal["low", "medium", "high"]
    revision: str
    document_owner: str
    approval_status: Literal["approved"]          # draft/obsolete procedures are never indexed


class GovernmentChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "government"
    industry: Literal["government"] = "government"
    agency_id: str
    record_class: str
    release_status: Literal["public_release", "restricted_release"]
    case_ref_token: str | None = Field(default=None, pattern=TOKEN)

    @model_validator(mode="after")
    def _release(self):
        if self.classification == "public" and self.release_status != "public_release":
            raise ValueError("public classification requires release_status=public_release")
        return self


class EducationHrChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "education_hr"
    industry: Literal["education_hr"] = "education_hr"
    domain: Literal["education", "hr"]
    org_unit: str
    course_or_department: str
    term: str | None = None
    region: str
    subject_ref_token: str | None = Field(default=None, pattern=TOKEN)


INDUSTRIES: dict[str, type[BaseChunk]] = {m.INDUSTRY: m for m in (
    HealthcareChunk, BankingChunk, InsuranceChunk, LegalChunk,
    RetailChunk, ManufacturingChunk, GovernmentChunk, EducationHrChunk)}

print("base fields:", ", ".join(BaseChunk.model_fields))
pd.DataFrame([{"industry": i, "model": m.__name__,
               "industry fields": ", ".join(f for f in m.model_fields if f not in BASE_FIELDS)}
              for i, m in INDUSTRIES.items()])

## 2. Embedding model — pinned, normalized, cached by content hash

In [ ]:
from sentence_transformers import SentenceTransformer


class Embedder:
    QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge v1.5 query instruction

    def __init__(self, model_name="BAAI/bge-small-en-v1.5", tag="bgesmall"):
        self.model_name, self.tag = model_name, tag
        self.model = SentenceTransformer(model_name)
        self.dim = (getattr(self.model, "get_embedding_dimension", None) or self.model.get_sentence_embedding_dimension)()
        self._cache, self.cache_hits = {}, 0

    def embed(self, texts):
        keys = [content_hash(t) for t in texts]
        todo = [(k, t) for k, t in zip(keys, texts) if k not in self._cache]
        self.cache_hits += len(texts) - len(todo)
        if todo:
            for (k, _), v in zip(todo, self.model.encode([t for _, t in todo], normalize_embeddings=True)):
                self._cache[k] = v.astype(float).tolist()
        return [self._cache[k] for k in keys]

    def embed_query(self, q):
        return self.model.encode(self.QUERY_PREFIX + q, normalize_embeddings=True).astype(float).tolist()


embedder = Embedder()
print(embedder.model_name, "| dim", embedder.dim)

## 3. Load Word / PDF with LangChain + attach the planned metadata

- `PyPDFLoader` (one `Document` per page) and `Docx2txtLoader` (one `Document` per file).
- `manifest.jsonl` is the source system's metadata export: tenant, ACL, classification, jurisdiction, validity, retention, industry fields. It is merged into `Document.metadata`.
- A file **without** a manifest entry is rejected: no metadata = no ACL = fail closed.
- **Clean** step: drop repeated header/footer lines (banner, address/footer bars, signature line) so they never get embedded.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)          # langchain-community sunset notice
    from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

LOADERS = {".pdf": PyPDFLoader, ".docx": Docx2txtLoader}
CLASS_WORDS = {"PUBLIC", "INTERNAL", "CONFIDENTIAL", "RESTRICTED"}


@dataclass
class Rejection:
    industry: str
    source_id: str
    gate: str
    reason: str


def clean_text(text: str) -> str:
    """Blueprint 'CLEAN' step: remove page furniture, keep content."""
    keep = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s in CLASS_WORDS:                       # empty / classification banner
            continue
        if "  |  " in s or s.startswith("Approved by:"):    # address bar, footer bar, signature line
            continue
        keep.append(re.sub(r"\s{2,}", " ", s))
    return "\n".join(keep)


def load_langchain_documents(industry: str):
    folder = DOCS / industry
    manifest = {m["file"]: m for m in (json.loads(l) for l in (folder / "manifest.jsonl").read_text("utf-8").splitlines() if l)}
    docs, rejections = [], []
    for path in sorted(p for p in folder.iterdir() if p.suffix.lower() in LOADERS):
        meta = manifest.get(path.name)
        if meta is None:
            rejections.append(Rejection(industry, path.stem, "metadata", f"{path.name} has no manifest entry")); continue
        loader = LOADERS[path.suffix.lower()]
        for page in loader(str(path)).load():
            page.metadata = {**page.metadata, **meta, "loader": loader.__name__}
            page.page_content = clean_text(page.page_content)
            docs.append(page)
    return docs, rejections


def to_source_docs(lc_docs: list[Document]) -> list[dict]:
    """Merge pages of the same file into one source doc (pipeline input)."""
    merged = {}
    for d in lc_docs:
        m = merged.setdefault(d.metadata["file"], {**{k: v for k, v in d.metadata.items()
                                                       if k not in ("page", "page_label", "total_pages")}, "text": ""})
        m["text"] = (m["text"] + "\n" + d.page_content).strip()
    return list(merged.values())


lc_docs, _ = load_langchain_documents("healthcare")
d0 = lc_docs[0]
print("page_content (cleaned):\n", d0.page_content[:600], "...\n")
{k: d0.metadata[k] for k in ["source", "loader", "file", "source_id", "tenant_id", "classification",
                             "allowed_groups", "jurisdiction", "valid_from", "retention_until", "metadata"]}

## 4. Ingestion gates (fail closed)

Order: **authorized source → approved purpose → classified → ACL present → retention valid → sensitive scan (reject secrets/SSN/cards, redact email/phone) → section-aware chunking (LangChain splitter) → embed → schema validation**. Each chunk text is prefixed with `title - section` so it stays meaningful on its own.

In [ ]:
APPROVED_SOURCES = {"ehr_export", "core_banking", "claims_system", "dms_legal", "pim", "plm", "records_mgmt", "lms_hris"}
APPROVED_PURPOSES = {"rag_search", "support_assistant"}
SECRETS = [re.compile(p) for p in (r"AKIA[0-9A-Z]{16}", r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
                                   r"(?i)\b(password|passwd|api[_-]?key|secret)\s*[:=]\s*\S+")]
SSN = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
CARD = re.compile(r"\b(?:\d[ -]?){13,19}\b")
EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b")
PHONE = re.compile(r"\+?\d{1,2}[ .-]?\(?\d{3}\)?[ .-]?\d{3}[ .-]?\d{4}\b")
SECTION = re.compile(r"^(\d+\. [A-Z][A-Za-z ]+|Revision History)$", re.M)

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40,
                                          separators=[". ", "; ", " ", ""], keep_separator="end")


def luhn(digits: str) -> bool:
    nums = [int(x) for x in digits][::-1]
    return sum(n if i % 2 == 0 else (n * 2 - 9 if n * 2 > 9 else n * 2) for i, n in enumerate(nums)) % 10 == 0


def sensitive_scan(text):
    """(reject_reason | None, redacted_text)"""
    if any(p.search(text) for p in SECRETS):
        return "credential/secret detected", text
    if SSN.search(text):
        return "SSN detected", text
    for m in CARD.finditer(text):
        digits = re.sub(r"\D", "", m.group())
        if 13 <= len(digits) <= 19 and luhn(digits):
            return "payment card number detected", text
    return None, PHONE.sub("[PHONE]", EMAIL.sub("[EMAIL]", text))


def split_sections(text, title):
    """[(section, body)] using the document's numbered headings; text before the first heading = control block."""
    marks = list(SECTION.finditer(text))
    parts = [("Document control", text[:marks[0].start()] if marks else text)]
    for i, m in enumerate(marks):
        parts.append((m.group(1), text[m.end(): marks[i + 1].start() if i + 1 < len(marks) else len(text)]))
    return [(s, " ".join(b.replace(title, "", 1).split() if s == "Document control" else b.split()))
            for s, b in parts if b.strip()]


@dataclass
class IngestResult:
    records: list = field(default_factory=list)
    rejections: list = field(default_factory=list)
    clean_docs: list = field(default_factory=list)


def run_pipeline(docs, industry, embedder) -> IngestResult:
    model, out = INDUSTRIES[industry], IngestResult()
    for doc in docs:
        sid = doc.get("source_id", "?")
        reject = lambda gate, why: out.rejections.append(Rejection(industry, sid, gate, why))
        if doc.get("source_system") not in APPROVED_SOURCES:
            reject("authorized_source", f"source_system '{doc.get('source_system')}' not approved"); continue
        if doc.get("purpose") not in APPROVED_PURPOSES:
            reject("approved_purpose", f"purpose '{doc.get('purpose')}' not approved"); continue
        if not doc.get("classification"):
            reject("classified", "missing classification"); continue
        if not doc.get("allowed_groups"):
            reject("acl", "no allowed_groups (ACL)"); continue
        if date.fromisoformat(doc["retention_until"]) < date.today():
            reject("retention", f"retention_until {doc['retention_until']} already passed"); continue
        why, text = sensitive_scan(doc["text"])
        if why:
            reject("sensitive_scan", why); continue

        pieces = [(sec, f"{doc['title']} - {sec}: {p}") for sec, body in split_sections(text, doc["title"])
                  for p in splitter.split_text(body)]
        vecs = embedder.embed([p for _, p in pieces])
        try:
            built = [model(
                chunk_id=make_chunk_id(doc["tenant_id"], sid, doc["source_version"], i),
                tenant_id=doc["tenant_id"], entity_type=doc["entity_type"], source_id=sid,
                source_version=doc["source_version"], source_file=doc["file"], chunk_no=i, section=sec,
                content_hash=content_hash(p), parent_id=sid, text=p, embedding=v,
                embedding_model=embedder.model_name, language=doc.get("language", "en"),
                classification=doc["classification"], allowed_groups=doc["allowed_groups"],
                jurisdiction=doc["jurisdiction"], valid_from=doc["valid_from"], valid_to=doc.get("valid_to"),
                retention_until=doc["retention_until"], **doc.get("metadata", {}))
                for i, ((sec, p), v) in enumerate(zip(pieces, vecs))]
        except ValidationError as e:
            err = e.errors()[0]
            reject("schema_validation", f"{'.'.join(map(str, err['loc'])) or 'record'}: {err['msg']}"); continue
        out.records += built
        out.clean_docs.append({**doc, "text": text})
    return out


demo = run_pipeline(to_source_docs(lc_docs), "healthcare", embedder)
pd.DataFrame([{"source_id": r.source_id, "chunk_no": r.chunk_no, "section": r.section, "text": r.text}
              for r in demo.records if r.source_id == "hc-001"])

## 5. LanceDB store — one table per industry

- **Schema**: the PyArrow schema is **generated from the Pydantic model** (`str`→`string`, `int`→`int64`, `date`→`date32`, `datetime`→`timestamp[us, UTC]`, `list[str]`→`list<string>`, `Literal`→`string`, embedding → `fixed_size_list<float32, 384>`). Arrow has no CHECK constraints, so `Literal`/ACL rules are enforced by Pydantic validation before any write (the gate in section 4).
- **Tables**: `{industry}_v1_bgesmall_384`. Scalar indexes: BTree on `chunk_id` (merge key), Bitmap on `tenant_id` and `status`, LabelList on `allowed_groups` (accelerates `array_has_any`).
- **No ANN index**: with < 100 chunks per table LanceDB's exact (brute-force) KNN is faster and exact; an IVF/HNSW vector index only pays off from roughly 10k+ rows (`tbl.create_index(metric="cosine", index_type="IVF_HNSW_SQ")`).
- **No alias**: LanceDB has no table alias/view. The blue/green switch point is the `CURRENT` dict (industry → versioned table), persisted in the SoR table `aliases`. Every function takes the *industry* and resolves it through `CURRENT`.
- **Upsert**: `merge_insert("chunk_id").when_matched_update_all().when_not_matched_insert_all()`. Like 01, `status` and `created_at` of already-stored chunks are **not** overwritten: they are re-read and carried over before the merge.
- **Search**: `.where(filter, prefilter=True)` → tenant + ACL + status + validity are applied **before** the KNN (see the query plan in section 9).
- **System of record**: SQLite `sources` + `quarantine` (+ `aliases`) in `data/sor/lancedb.db`, redacted text only.

In [ ]:
import sqlite3
from datetime import timedelta
import lancedb
import pyarrow as pa
from lancedb.index import BTree, Bitmap, LabelList

LANCE_DIR = ROOT / "data" / "lancedb"
SOR_PATH = ROOT / "data" / "sor" / "lancedb.db"
PA_SCALAR = {str: pa.string(), int: pa.int64(), date: pa.date32(), float: pa.float64(),
             datetime: pa.timestamp("us", tz="UTC")}
CURRENT: dict[str, str] = {}          # alias substitute: industry -> active versioned table
_TABLES: dict[str, "lancedb.table.Table"] = {}


def index_name(industry):
    return f"{industry}_v{SCHEMA_VERSION}_{embedder.tag}_{embedder.dim}"


def arrow_field(name, f):
    if name == "embedding":
        return pa.field("embedding", pa.list_(pa.float32(), embedder.dim), nullable=False)
    ann, nullable = f.annotation, not (f.is_required() or f.default is not None)
    if get_origin(ann) in (Union, types.UnionType):
        ann, nullable = [a for a in get_args(ann) if a is not type(None)][0], True
    if get_origin(ann) is Literal:
        typ = pa.string()
    elif get_origin(ann) is list:
        typ = pa.list_(PA_SCALAR[get_args(ann)[0]])
    else:
        typ = PA_SCALAR[ann]
    return pa.field(name, typ, nullable=nullable)


def arrow_schema(model):
    return pa.schema([arrow_field(n, f) for n, f in model.model_fields.items()])


SOR_DDL = """
CREATE TABLE IF NOT EXISTS sources (
  source_id TEXT NOT NULL, source_version TEXT NOT NULL, tenant_id TEXT NOT NULL, industry TEXT NOT NULL,
  title TEXT, source_file TEXT, text TEXT NOT NULL, raw TEXT NOT NULL, retention_until TEXT NOT NULL,
  ingested_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP, PRIMARY KEY (source_id, source_version));
CREATE TABLE IF NOT EXISTS quarantine (
  id INTEGER PRIMARY KEY AUTOINCREMENT, industry TEXT, source_id TEXT, gate TEXT, reason TEXT,
  at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP, UNIQUE (industry, source_id, gate, reason));
CREATE TABLE IF NOT EXISTS aliases (industry TEXT PRIMARY KEY, table_name TEXT NOT NULL);
"""


def lit(v: str) -> str:
    """SQL string literal for LanceDB/DataFusion filters (quotes escaped)."""
    return "'" + str(v).replace("'", "''") + "'"


def in_list(values) -> str:
    return "(" + ", ".join(lit(v) for v in values) + ")"


def connect():
    LANCE_DIR.mkdir(parents=True, exist_ok=True)
    SOR_PATH.parent.mkdir(parents=True, exist_ok=True)
    sor = sqlite3.connect(SOR_PATH, isolation_level=None)       # autocommit
    return lancedb.connect(LANCE_DIR), sor


def table(industry):
    return _TABLES[CURRENT[industry]]


def create_indexes(db, sor, fresh=True):
    """fresh=True drops and rebuilds tables + SoR so the notebook is re-runnable; fresh=False is idempotent (exist_ok)."""
    if fresh:
        sor.executescript("DROP TABLE IF EXISTS sources; DROP TABLE IF EXISTS quarantine; DROP TABLE IF EXISTS aliases;")
    sor.executescript(SOR_DDL)
    for ind, model in INDUSTRIES.items():
        t = index_name(ind)
        if fresh:
            db.drop_table(t, ignore_missing=True)
        tbl = db.create_table(t, schema=arrow_schema(model), exist_ok=True)
        tbl.create_index("chunk_id", config=BTree(), replace=True)
        tbl.create_index("tenant_id", config=Bitmap(), replace=True)
        tbl.create_index("status", config=Bitmap(), replace=True)
        tbl.create_index("allowed_groups", config=LabelList(), replace=True)
        _TABLES[t] = tbl
        CURRENT[ind] = t
        sor.execute("INSERT INTO aliases (industry, table_name) VALUES (?, ?) "
                    "ON CONFLICT (industry) DO UPDATE SET table_name = excluded.table_name", (ind, t))
    return dict(CURRENT)


def save_sources(sor, industry, docs):
    sor.executemany(
        "INSERT INTO sources (source_id, source_version, tenant_id, industry, title, source_file, text, raw, retention_until) "
        "VALUES (?,?,?,?,?,?,?,?,?) ON CONFLICT (source_id, source_version) DO UPDATE "
        "SET text = excluded.text, raw = excluded.raw, retention_until = excluded.retention_until",
        [(d["source_id"], d["source_version"], d["tenant_id"], industry, d.get("title"), d.get("file"), d["text"],
          json.dumps({k: v for k, v in d.items() if k != "text"}, default=str), d["retention_until"]) for d in docs])


def save_rejections(sor, rejections):
    sor.executemany("INSERT OR IGNORE INTO quarantine (industry, source_id, gate, reason) VALUES (?,?,?,?)",
                    [(r.industry, r.source_id, r.gate, r.reason) for r in rejections])


def upsert(db, industry, records):
    """Idempotent merge on chunk_id; status/created_at of existing chunks are carried over, never reset."""
    if not records:
        return None
    tbl, ids = table(industry), [r.chunk_id for r in records]
    existing = {}
    for i in range(0, len(ids), 500):
        batch = ids[i:i + 500]
        for row in (tbl.search().where(f"chunk_id IN {in_list(batch)}")
                    .select(["chunk_id", "status", "created_at"]).limit(len(batch)).to_list()):
            existing[row["chunk_id"]] = row
    rows = []
    for r in records:
        d = r.model_dump()
        if r.chunk_id in existing:
            d["status"], d["created_at"] = existing[r.chunk_id]["status"], existing[r.chunk_id]["created_at"]
        rows.append(d)
    data = pa.Table.from_pylist(rows, schema=arrow_schema(type(records[0])))
    return (tbl.merge_insert("chunk_id").when_matched_update_all().when_not_matched_insert_all().execute(data))


def promote(db, industry, source_ids=None):
    where = "status = 'staged'" + (f" AND source_id IN {in_list(source_ids)}" if source_ids else "")
    return table(industry).update(where=where, values={"status": "active"}).rows_updated


def supersede(db, industry, source_id, keep_version):
    return table(industry).update(
        where=f"source_id = {lit(source_id)} AND source_version <> {lit(keep_version)} AND status = 'active'",
        values={"status": "superseded"}).rows_updated


def purge(db, sor, industry, source_id):
    """Hard erase: delete rows, then drop old table versions (time travel) so the data is gone from disk too."""
    tbl = table(industry)
    n = tbl.delete(f"source_id = {lit(source_id)}").num_deleted_rows
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)   # single-process demo: no concurrent readers of old versions
        tbl.optimize(cleanup_older_than=timedelta(0))
    sor.execute("DELETE FROM sources WHERE source_id = ?", (source_id,))
    return n


def count(db, industry, where=None):
    return table(industry).count_rows(where)


def safe_filter(tenant_id, groups):
    return (f"tenant_id = {lit(tenant_id)} AND array_has_any(allowed_groups, [{', '.join(map(lit, groups))}]) "
            f"AND status = 'active' AND valid_from <= current_date AND (valid_to IS NULL OR valid_to >= current_date)")


SEARCH_COLS = ["chunk_id", "tenant_id", "source_id", "source_version", "section", "text"]


def search_query(industry, tenant_id, groups, query_vec, k=5):
    return (table(industry).search(np.asarray(query_vec, dtype=np.float32), vector_column_name="embedding")
            .distance_type("cosine").where(safe_filter(tenant_id, groups), prefilter=True)
            .select(SEARCH_COLS + ["_distance"]).limit(k))


def safe_search(db, industry, tenant_id, groups, query_vec, k=5):
    """Tenant + ACL (any-match) + status + validity as a PRE-filter inside the LanceDB query, then exact cosine KNN."""
    if not tenant_id or not groups:
        raise PermissionError("safe_search needs tenant_id and at least one group")
    rows = search_query(industry, tenant_id, groups, query_vec, k).to_list()
    return [{**{c: r[c] for c in SEARCH_COLS}, "score": 1 - r["_distance"]} for r in rows]


db, sor = connect()
print("lancedb", lancedb.__version__, "| pyarrow", pa.__version__, "| dir", LANCE_DIR, "| SoR", SOR_PATH)
print(arrow_schema(HealthcareChunk))

In [ ]:
tables = create_indexes(db, sor, fresh=True)
pd.DataFrame([{"industry (CURRENT key)": i, "table": t,
               "scalar indexes": ", ".join(f"{ix.columns[0]}:{ix.index_type}" for ix in table(i).list_indices())}
              for i, t in tables.items()])

## 6. Upload all industry documents (Word/PDF → LangChain → gates → LanceDB)

In [ ]:
results, summary = {}, []
for ind in INDUSTRIES:
    lc, missing_meta = load_langchain_documents(ind)
    docs = to_source_docs(lc)
    res = run_pipeline(docs, ind, embedder)
    res.rejections += missing_meta
    save_sources(sor, ind, res.clean_docs)
    upsert(db, ind, res.records)
    save_rejections(sor, res.rejections)
    table(ind).optimize()                     # fold new rows into the scalar indexes
    results[ind] = res
    summary.append({"industry": ind, "files": len(docs),
                    "pdf": sum(d["file"].endswith(".pdf") for d in docs), "docx": sum(d["file"].endswith(".docx") for d in docs),
                    "accepted": len(res.clean_docs), "chunks": len(res.records), "quarantined": len(res.rejections),
                    "tenants": ", ".join(sorted({d["tenant_id"] for d in docs}))})
summary_df = pd.DataFrame(summary)
assert (summary_df["quarantined"] >= 1).all() and all(count(db, i) == len(results[i].records) for i in INDUSTRIES)
summary_df

In [ ]:
# quarantine (fail closed): every industry's bad document stopped at a gate (SQLite system of record)
pd.DataFrame(sor.execute("select industry, source_id, gate, reason from quarantine order by industry").fetchall(),
             columns=["industry", "source_id", "gate", "reason"])

In [ ]:
# stored rows include the industry-specific columns (typed Arrow columns)
cols = ["source_id", "source_file", "chunk_no", "section", "tenant_id", "record_type", "patient_ref_token",
        "consent_scope", "encounter_date", "allowed_groups", "status"]
table("healthcare").to_pandas()[cols].sort_values(["source_id", "chunk_no"]).reset_index(drop=True)

In [ ]:
# redaction: the email inside hc-001.pdf never reaches the index or the system of record
txt = sor.execute("select text from sources where source_id = 'hc-001'").fetchone()[0]
assert "@" not in txt and "[EMAIL]" in txt
assert not table("healthcare").to_pandas()["text"].str.contains("@").any()
print(re.search(r"[^.]*\[EMAIL\][^.]*\.?", txt).group().strip())

## 7. Idempotency — same upload again, same row counts

In [ ]:
before = {i: count(db, i) for i in INDUSTRIES}
merges = {ind: upsert(db, ind, res.records) for ind, res in results.items()}
after = {i: count(db, i) for i in INDUSTRIES}
assert before == after
assert all(m.num_inserted_rows == 0 for m in merges.values())
assert all(count(db, i, "status <> 'staged'") == 0 for i in INDUSTRIES)      # nothing promoted yet, nothing reset
print("idempotent ✔", after, "| inserted on re-run:", sum(m.num_inserted_rows for m in merges.values()),
      "| embedding cache hits:", embedder.cache_hits)

## 8. Staging → promote — staged chunks are never searchable

In [ ]:
golden = json.loads((SAMPLES / "golden_queries.json").read_text())
staged_hits = {g["query"]: len(safe_search(db, g["industry"], g["tenant_id"], g["groups"], embedder.embed_query(g["query"])))
               for g in golden}
assert sum(staged_hits.values()) == 0
print("hits while staged (all 16 golden queries):", sum(staged_hits.values()))
print("promoted:", {i: promote(db, i) for i in INDUSTRIES})
assert all(count(db, i, "status = 'staged'") == 0 for i in INDUSTRIES)
# re-uploading the same documents after promotion must NOT reset active chunks back to staged
active_before = {i: count(db, i, "status = 'active'") for i in INDUSTRIES}
remerge = {i: upsert(db, i, results[i].records) for i in INDUSTRIES}
assert all(m.num_inserted_rows == 0 for m in remerge.values())
assert {i: count(db, i, "status = 'active'") for i in INDUSTRIES} == active_before
assert all(count(db, i, "status = 'staged'") == 0 for i in INDUSTRIES)
print("re-upload after promote keeps status=active ✔")
pd.DataFrame([{"industry": i, "status": s, "chunks": count(db, i, f"status = {lit(s)}")}
              for i in INDUSTRIES for s in ("staged", "active") if count(db, i, f"status = {lit(s)}")])

## 9. Safe search — golden queries per industry

In [ ]:
rows, latencies = [], []
for g in golden:
    qv = embedder.embed_query(g["query"])
    t0 = time.perf_counter()
    hits = safe_search(db, g["industry"], g["tenant_id"], g["groups"], qv, k=5)
    latencies.append((time.perf_counter() - t0) * 1000)
    found = [h["source_id"] for h in hits]
    rows.append({"industry": g["industry"], "tenant": g["tenant_id"], "query": g["query"], "expected": g["expected_source_id"],
                 "top1": found[0] if found else None, "top1 section": hits[0]["section"] if hits else None,
                 "score": round(hits[0]["score"], 3) if hits else None,
                 "hit@1": found[:1] == [g["expected_source_id"]], "hit@5": g["expected_source_id"] in found})
search_df = pd.DataFrame(rows)
assert search_df["hit@5"].all(), search_df[~search_df["hit@5"]]
print(f"hit@1 = {search_df['hit@1'].mean():.2f} | hit@5 = {search_df['hit@5'].mean():.2f} | p50 latency = {statistics.median(latencies):.1f} ms")
search_df

In [ ]:
# the query plan proves the pre-filter: FilterExec (tenant/ACL/status/dates) runs BELOW KNNVectorDistance
qv = embedder.embed_query("how to lock out the hydraulic press")
plan = search_query("manufacturing", "forgeworks", ["maintenance"], qv).explain_plan(verbose=False)
lines = plan.splitlines()
knn = next(i for i, l in enumerate(lines) if "KNNVectorDistance" in l)
assert any("tenant_id" in l for l in lines[knn + 1:]), "tenant filter must run below (before) the KNN"
print(plan)
pd.DataFrame(safe_search(db, "manufacturing", "forgeworks", ["maintenance"], qv))

## 10. Isolation tests
LanceDB is an **embedded library: no server, no users, no roles, no RLS, no native multi-tenancy**. Anyone holding the table handle can read every tenant's rows. Isolation is therefore **filter-only**: `safe_search` is the only read path and it always adds `tenant_id = … AND array_has_any(allowed_groups, …)` as a pre-filter. (Stronger options: one table / one directory per tenant, or OS/object-store permissions per tenant path.)

1. **Filter-only probe**: an unfiltered query sees several tenants (the DB does not isolate); the tenant pre-filter returns exactly that tenant's rows.
2. **Cross-tenant**: other tenant asks the same golden question → never gets the owner's chunks.
3. **ACL**: user without the group gets nothing. 4. **Fail closed**: no tenant/groups → `PermissionError`.

In [ ]:
probe = []
for ind in INDUSTRIES:
    tenants = sorted({d["tenant_id"] for d in results[ind].clean_docs})
    qv = embedder.embed_query("summary")
    raw = table(ind).search(np.asarray(qv, dtype=np.float32), vector_column_name="embedding").select(["tenant_id", "_distance"]).limit(1000).to_list()
    row = {"industry": ind, "total rows": count(db, ind), "tenants seen by UNFILTERED query": len({r["tenant_id"] for r in raw})}
    row |= {f"rows tenant_id={t}": count(db, ind, f"tenant_id = {lit(t)}") for t in tenants}
    assert sum(count(db, ind, f"tenant_id = {lit(t)}") for t in tenants) == row["total rows"]
    for t in tenants:   # pre-filtered results never contain another tenant
        groups = sorted({g for d in results[ind].clean_docs if d["tenant_id"] == t for g in d["allowed_groups"]})
        assert all(h["tenant_id"] == t for h in safe_search(db, ind, t, groups, qv, k=50))
    probe.append(row)
probe_df = pd.DataFrame(probe).astype(object).fillna("-")
assert (probe_df["tenants seen by UNFILTERED query"] >= 2).any()     # honest: the DB itself does not isolate
probe_df

In [ ]:
iso = []
for g in golden:
    other = ({d["tenant_id"] for d in results[g["industry"]].clean_docs} - {g["tenant_id"]}).pop()
    qv = embedder.embed_query(g["query"])
    hits = safe_search(db, g["industry"], other, g["groups"], qv)
    leaked = [h for h in hits if h["tenant_id"] != other or h["source_id"] == g["expected_source_id"]]
    no_acl = safe_search(db, g["industry"], g["tenant_id"], ["group_nobody_has"], qv)
    iso.append({"industry": g["industry"], "owner": g["tenant_id"], "asked as": other,
                "hits for other tenant": len(hits), "leaked": len(leaked), "hits without ACL group": len(no_acl)})
iso_df = pd.DataFrame(iso)
assert iso_df["leaked"].eq(0).all() and iso_df["hits without ACL group"].eq(0).all()
for bad in [("", ["care_team_cardiology"]), ("acme_health", [])]:
    try:
        safe_search(db, "healthcare", *bad, embedder.embed_query("x")); raise AssertionError("should raise")
    except PermissionError as e:
        print("fail closed ✔", e)
iso_df

## 11. Update — new version of `hc-002.docx` supersedes v1

In [ ]:
v1 = next(d for d in to_source_docs(load_langchain_documents("healthcare")[0]) if d["source_id"] == "hc-002")
v2 = {**v1, "source_version": "2",
      "text": v1["text"].replace("Metformin dose increased to 1000 mg twice daily", "Metformin dose increased to 1500 mg extended release once daily")}
res2 = run_pipeline([v2], "healthcare", embedder)
save_sources(sor, "healthcare", res2.clean_docs)
upsert(db, "healthcare", res2.records)
promote(db, "healthcare", ["hc-002"])
n_sup = supersede(db, "healthcare", "hc-002", keep_version="2")
print("superseded chunks:", n_sup)
hits = safe_search(db, "healthcare", "acme_health", ["care_team_endocrine"], embedder.embed_query("metformin dose"))
assert n_sup > 0 and hits and any(h["source_id"] == "hc-002" for h in hits)
assert all(h["source_version"] == "2" for h in hits if h["source_id"] == "hc-002")
assert count(db, "healthcare", "source_id = 'hc-002' AND source_version = '1' AND status = 'active'") == 0
hc2 = table("healthcare").to_pandas().query("source_id == 'hc-002'")
display(hc2.groupby(["source_version", "status"]).size().rename("chunks").reset_index())
pd.DataFrame(hits)[["source_id", "source_version", "section", "score", "text"]]

## 12. Delete / consent withdrawal — purge `hc-004` and verify absence

In [ ]:
q = embedder.embed_query("ejection fraction systolic dysfunction")
before = [h["source_id"] for h in safe_search(db, "healthcare", "beacon_clinic", ["care_team_cardiology"], q)]
purged = purge(db, sor, "healthcare", "hc-004")
after = [h["source_id"] for h in safe_search(db, "healthcare", "beacon_clinic", ["care_team_cardiology"], q)]
left = sor.execute("select count(*) from sources where source_id = 'hc-004'").fetchone()[0]
left_vec = count(db, "healthcare", "source_id = 'hc-004'")
assert "hc-004" in before and "hc-004" not in after and left == 0 and left_vec == 0 and purged > 0
print(f"purged {purged} chunks | before: {sorted(set(before))} | after: {sorted(set(after))} | "
      f"SoR rows left: {left} | LanceDB rows left: {left_vec} ✔")
# LanceDB keeps old table versions (time travel); purge() runs optimize(cleanup_older_than=0) so no surviving
# version still holds hc-004 -- check every remaining version, not just the latest
tbl = table("healthcare")
versions = [v["version"] for v in tbl.list_versions()]
in_old = {}
for v in versions:
    tbl.checkout(v)
    in_old[v] = tbl.count_rows("source_id = 'hc-004'")
tbl.checkout_latest()
assert sum(in_old.values()) == 0, in_old
assert count(db, "healthcare", "source_id = 'hc-004'") == 0
print("table versions kept after purge:", versions, "| rows of hc-004 in any version:", sum(in_old.values()), "✔")

## 13. Metrics (for the cross-DB comparison notebook)

In [ ]:
metrics = {"db": "lancedb", "industries": len(INDUSTRIES), "chunks_total": sum(count(db, i) for i in INDUSTRIES),
           "quarantined": sum(len(r.rejections) for r in results.values()),
           "hit@1": float(search_df["hit@1"].mean()), "hit@5": float(search_df["hit@5"].mean()),
           "latency_p50_ms": round(statistics.median(latencies), 2),
           "isolation": "filter-only (embedded, no auth/RLS/native tenancy): tenant_id + array_has_any(allowed_groups) "
                        "+ status/validity pre-filter in every query; PASS 0 leaks",
           "executed": True}
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "lancedb.json").write_text(json.dumps(metrics, indent=2))
metrics

## Next: embedding model change (blue/green)
Never re-embed into the existing tables. Create `{industry}_v1_<newmodel>_<dim>` (schema from `arrow_schema` with the new `Embedder`), re-embed from the SQLite `sources` table (system of record), run the golden queries against it, then flip the pointer: `CURRENT[industry] = new_table` and `UPDATE aliases SET table_name = ? WHERE industry = ?`. LanceDB has no alias/view, so this pointer is the switch; drop the old table once nothing reads it.